# RAG retrieval evaluation

Most RAG write-ups report an answer-quality score and stop there. That number
can't tell you whether a bad answer came from the retriever missing the chunk or
the model ignoring it, so I built this to measure retrieval on its own.

The corpus is 25 pages of FastAPI documentation, about 35,000 words. I wrote 52
questions against it and split them into four groups — wording that matches the
source, paraphrased wording, indirect questions needing two ideas connected, and
questions the corpus can't answer — because a single aggregate number hid the
effect I was most interested in.

BM25 and TF-IDF are implemented here rather than imported. No vector database, no
API calls.

## 1. Setup

In [1]:
try:
    import sklearn, numpy
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "numpy"])

import json, math, re, random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(0)
np.random.seed(0)
print("ready")

ready


In [11]:
import urllib.request, pathlib
BASE = "https://raw.githubusercontent.com/fastapi/fastapi/master/docs/en/docs"
PAGES = ["async","deployment/concepts","deployment/docker","deployment/https",
 "tutorial/background-tasks","tutorial/bigger-applications","tutorial/body",
 "tutorial/body-nested-models","tutorial/cookie-params","tutorial/cors",
 "tutorial/debugging","tutorial/dependencies/index","tutorial/handling-errors",
 "tutorial/header-params","tutorial/metadata","tutorial/middleware",
 "tutorial/path-operation-configuration","tutorial/path-params",
 "tutorial/query-params","tutorial/request-files","tutorial/response-model",
 "tutorial/security/first-steps","tutorial/sql-databases",
 "tutorial/static-files","tutorial/testing"]
out = pathlib.Path("corpus"); out.mkdir(exist_ok=True)
for p in PAGES:
    (out / (p.replace("/", "_") + ".md")).write_bytes(
        urllib.request.urlopen(f"{BASE}/{p}.md", timeout=20).read())
print(len(list(out.glob("*.md"))), "files")

25 files


In [12]:
import os
files = sorted(os.listdir("corpus"))
print(os.getcwd(), len(files))
print([(f, os.path.getsize(f"corpus/{f}")) for f in files[:5]])

D:\chrome downloads\rag-retrieval-eval-dense\rag-retrieval-eval 25
[('async.md', 23978), ('deployment_concepts.md', 18936), ('deployment_docker.md', 28940), ('deployment_https.md', 14304), ('tutorial_background-tasks.md', 4786)]


## 2. Corpus

I used FastAPI's documentation because it's real technical prose with consistent
terminology, and I know it well enough to write questions against it. 25 pages,
fetched by `fetch_corpus.py` rather than committed.

In [13]:
@dataclass
class Doc:
    doc_id: str
    title: str
    text: str

def load_folder(path: str) -> list[Doc]:
    """One file = one document. Markdown is left as-is; the retrievers tokenize."""
    docs = []
    for p in sorted(Path(path).rglob("*")):
        if p.suffix.lower() in {".txt", ".md"}:
            body = p.read_text(encoding="utf-8", errors="ignore")
            docs.append(Doc(p.stem, p.stem, re.sub(r"\s+", " ", body).strip()))
    return docs

docs = load_folder("corpus")
if not docs:
    raise SystemExit("corpus/ is empty — run `python fetch_corpus.py` first")

lens = [len(d.text.split()) for d in docs]
print(f"{len(docs)} documents, {sum(lens):,} words")
print(f"shortest {min(lens)} words, longest {max(lens)}, median {int(np.median(lens))}")

25 documents, 35,115 words
shortest 244 words, longest 4238, median 993


## 3. Eval set

52 questions, each paired with the document that answers it.

I grouped them deliberately. The overlap group reuses the document's own
vocabulary, which is the easy case. The paraphrased group asks the same things in
words a user would actually type. The indirect group needs two ideas connected.
Six questions have no answer in the corpus at all, to see what the retriever does
when there's nothing to find.

Splitting them this way is what surfaced the result in section 9.

In [14]:
EVAL = [
    ("How do I declare a request body?", "tutorial_body"),
    ("How do I declare query parameters?", "tutorial_query-params"),
    ("How do I declare path parameters with types?", "tutorial_path-params"),
    ("How do I use a response model?", "tutorial_response-model"),
    ("How do I raise an HTTPException?", "tutorial_handling-errors"),
    ("What is dependency injection in FastAPI?", "tutorial_dependencies_index"),
    ("How do I create a middleware?", "tutorial_middleware"),
    ("How do I configure CORS?", "tutorial_cors"),
    ("How do I add background tasks?", "tutorial_background-tasks"),
    ("How do I serve static files?", "tutorial_static-files"),
    ("How do I declare cookie parameters?", "tutorial_cookie-params"),
    ("How do I declare header parameters?", "tutorial_header-params"),
    ("How do I upload files?", "tutorial_request-files"),
    ("How do I test my application?", "tutorial_testing"),
    ("How do I run FastAPI in Docker?", "deployment_docker"),
    ("What is async and await?", "async"),

    # --- paraphrased: little or no shared vocabulary -----------------------
    ("Why does my browser block calls to my API from another site?", "tutorial_cors"),
    ("How do I stop a password field leaking in the JSON I send back?", "tutorial_response-model"),
    ("I want to send an email after replying to the user without making them wait", "tutorial_background-tasks"),
    ("How do I return a 404 when a record is not found?", "tutorial_handling-errors"),
    ("How do I avoid repeating the same auth check in every endpoint?", "tutorial_dependencies_index"),
    ("My project is one huge file, how do I break it up?", "tutorial_bigger-applications"),
    ("How do I read a value the client stored in the browser and sends back?", "tutorial_cookie-params"),
    ("How do I time every request and add the duration to the reply?", "tutorial_middleware"),
    ("How do I connect my app to a database?", "tutorial_sql-databases"),
    ("How do I put a breakpoint in and step through my code?", "tutorial_debugging"),
    ("What do I write to accept a JSON object from the client?", "tutorial_body"),
    ("Should my endpoint function be regular or the other kind?", "async"),
    ("How do I change the title shown on the interactive docs page?", "tutorial_metadata"),
    ("How do I get a certificate so my site is encrypted?", "deployment_https"),
    ("How do I check my code works without starting a server?", "tutorial_testing"),
    ("Where do I put my CSS and images?", "tutorial_static-files"),
    ("How do I accept a list of items inside the payload?", "tutorial_body-nested-models"),
    ("How do I set which HTTP code comes back on success?", "tutorial_path-operation-configuration"),
    ("How do I read a token the client puts in the request metadata?", "tutorial_header-params"),

    # --- indirect: requires connecting two ideas ---------------------------
    ("Why would running my app in one process not use all my CPU cores?", "deployment_concepts"),
    ("What happens if two middlewares are added, which runs first?", "tutorial_middleware"),
    ("How does the framework know a URL segment should be an integer?", "tutorial_path-params"),
    ("What does the framework do with a value that is not in my model?", "tutorial_response-model"),
    ("Why is a container image different from a running container?", "deployment_docker"),
    ("How do I group endpoints under a label in the docs?", "tutorial_path-operation-configuration"),
    ("What does OAuth2 give me as a first step?", "tutorial_security_first-steps"),
    ("Why do I need a domain name before getting encryption working?", "deployment_https"),
    ("How do dependencies apply to a whole group of routes at once?", "tutorial_bigger-applications"),
    ("What happens to a value the client sends that my endpoint does not declare?", "tutorial_query-params"),
    ("Why might blocking code slow down every other request?", "async"),

    # --- unanswerable: nothing in the corpus covers these ------------------
    ("How do I deploy to Kubernetes with a Helm chart?", None),
    ("What is the pricing for FastAPI enterprise support?", None),
    ("How do I train a machine learning model on this data?", None),
    ("How do I configure Redis caching layers?", None),
    ("What is the recommended way to write Django migrations?", None),
    ("How do I set up a GraphQL subscription?", None),
]

answerable = [(q, d) for q, d in EVAL if d]
print(f"{len(EVAL)} questions, {len(answerable)} answerable, {len(EVAL)-len(answerable)} unanswerable")
print(f"covering {len(set(d for _, d in answerable))} of {len(docs)} documents")

52 questions, 46 answerable, 6 unanswerable
covering 25 of 25 documents


## 4. Chunking

Retrieval runs on chunks, so where the boundaries fall decides what can be found
at all.

I compared two approaches. Fixed splits every N words with overlap, which is
simple but cuts sentences in half. Sentence-aware packs whole sentences up to a
budget. The overlap in the fixed version exists because an answer sitting on a
boundary would otherwise be unreachable from either side.

In [15]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str

def chunk_fixed(docs, size=60, overlap=15) -> list[Chunk]:
    out = []
    for d in docs:
        words = d.text.split()
        step = max(1, size - overlap)
        for i in range(0, len(words), step):
            piece = words[i:i + size]
            if len(piece) < 10 and out and out[-1].doc_id == d.doc_id:
                break                       # trailing scrap, skip it
            out.append(Chunk(f"{d.doc_id}#{i}", d.doc_id, " ".join(piece)))
    return out

def chunk_sentences(docs, budget=60) -> list[Chunk]:
    out = []
    for d in docs:
        sents = re.split(r"(?<=[.!?])\s+", d.text)
        buf, n, idx = [], 0, 0
        for s in sents:
            w = len(s.split())
            if buf and n + w > budget:
                out.append(Chunk(f"{d.doc_id}#s{idx}", d.doc_id, " ".join(buf)))
                idx += 1
                buf, n = [], 0
            buf.append(s); n += w
        if buf:
            out.append(Chunk(f"{d.doc_id}#s{idx}", d.doc_id, " ".join(buf)))
    return out

for name, fn in [("fixed", chunk_fixed), ("sentence", chunk_sentences)]:
    ch = fn(docs)
    sizes = [len(c.text.split()) for c in ch]
    print(f"{name:<9} {len(ch):>3} chunks   mean {np.mean(sizes):.0f} words   "
          f"min {min(sizes)}  max {max(sizes)}")

fixed     787 chunks   mean 59 words   min 10  max 60
sentence  723 chunks   mean 49 words   min 9  max 132


## 5. Retrievers

BM25 ranks lexically, weighting rare terms and normalising for document length.
TF-IDF ranks by cosine similarity over term vectors. The hybrid fuses both by
rank rather than by score, so the two scales don't have to be comparable.

I wrote BM25 out instead of importing `rank_bm25` so the k1 and b parameters are
visible and I could reason about what the scoring was doing to long documents.

In [16]:
def tokenize(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", s.lower())

class BM25:
    """Okapi BM25. k1 damps term frequency, b controls length normalisation."""
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.k1, self.b = k1, b
        self.toks = [tokenize(c.text) for c in chunks]
        self.len = [len(t) for t in self.toks]
        self.avg = sum(self.len) / len(self.len)
        self.tf = [Counter(t) for t in self.toks]
        df = Counter()
        for t in self.toks:
            df.update(set(t))
        N = len(chunks)
        self.idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}

    def search(self, query, k=5):
        q = tokenize(query)
        scores = []
        for i, tf in enumerate(self.tf):
            s = 0.0
            for w in q:
                if w not in tf:
                    continue
                f = tf[w]
                s += self.idf[w] * f * (self.k1 + 1) / (
                    f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
            scores.append(s)
        order = np.argsort(scores)[::-1][:k]
        return [(self.chunks[i], scores[i]) for i in order]

class TfidfRetriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vec = TfidfVectorizer(stop_words="english", sublinear_tf=True)
        self.M = self.vec.fit_transform([c.text for c in chunks])

    def search(self, query, k=5):
        sims = cosine_similarity(self.vec.transform([query]), self.M)[0]
        order = np.argsort(sims)[::-1][:k]
        return [(self.chunks[i], sims[i]) for i in order]

class Hybrid:
    """Reciprocal rank fusion: score = sum over retrievers of 1/(rank + c).
    Uses ranks not scores, so BM25 and cosine need no normalisation."""
    def __init__(self, retrievers, c=60):
        self.rs, self.c = retrievers, c

    def search(self, query, k=5):
        fused = defaultdict(float)
        seen = {}
        for r in self.rs:
            for rank, (chunk, _) in enumerate(r.search(query, k=k * 3)):
                fused[chunk.chunk_id] += 1 / (self.c + rank + 1)
                seen[chunk.chunk_id] = chunk
        top = sorted(fused.items(), key=lambda kv: -kv[1])[:k]
        return [(seen[cid], sc) for cid, sc in top]

chunks = chunk_sentences(docs)
bm25, tfidf = BM25(chunks), TfidfRetriever(chunks)
hybrid = Hybrid([bm25, tfidf])

for chunk, score in bm25.search("can contractors connect remotely", k=3):
    print(f"{score:.2f}  {chunk.doc_id:<9} {chunk.text[:70]}...")

9.00  tutorial_debugging # Debugging { #debugging } You can connect the debugger in your editor...
5.57  deployment_docker So, you would run **multiple containers** with different things, like ...
4.76  tutorial_sql-databases You would have **one single `engine` object** for all your code to con...


## 6. Metrics

- `recall@k` — is the right document in the top k. This caps everything
  downstream, since a chunk that isn't retrieved can't be used no matter how good
  the model is.
- `MRR` — how high it ranked. Matters because context windows are finite and
  models weight early context more heavily.
- `hit@1` — was the top result right. The strictest read.

In [17]:
def evaluate(retriever, eval_set, k=5):
    recall = mrr = hit1 = 0
    misses = []
    qs = [(q, d) for q, d in eval_set if d]

    for q, gold in qs:
        got = [c.doc_id for c, _ in retriever.search(q, k=k)]
        if gold in got:
            recall += 1
            rank = got.index(gold) + 1
            mrr += 1 / rank
            if rank == 1:
                hit1 += 1
        else:
            misses.append((q, gold, got[:3]))

    n = len(qs)
    return {"recall@k": recall / n, "mrr": mrr / n, "hit@1": hit1 / n,
            "n": n, "misses": misses}

r = evaluate(bm25, EVAL)
print(f"BM25   recall@5 {r['recall@k']:.1%}   MRR {r['mrr']:.3f}   hit@1 {r['hit@1']:.1%}")

BM25   recall@5 67.4%   MRR 0.568   hit@1 50.0%


## 7. Chunking experiment

Retriever held fixed at BM25, chunking varied. One variable at a time, against a
declared baseline, so any difference is attributable.

In [18]:
def run(chunker, retriever_cls, label, k=5, **kw):
    ch = chunker(docs, **kw)
    ret = retriever_cls(ch)
    r = evaluate(ret, EVAL, k=k)
    return {"config": label, "chunks": len(ch), **{x: r[x] for x in ("recall@k","mrr","hit@1")}}

rows = [
    run(chunk_fixed,     BM25, "fixed-40/10",  size=40, overlap=10),
    run(chunk_fixed,     BM25, "fixed-60/15",  size=60, overlap=15),
    run(chunk_fixed,     BM25, "fixed-100/20", size=100, overlap=20),
    run(chunk_sentences, BM25, "sentence-40",  budget=40),
    run(chunk_sentences, BM25, "sentence-60",  budget=60),
    run(chunk_sentences, BM25, "sentence-100", budget=100),
]

print(f"{'config':<15}{'chunks':>7}{'recall@5':>10}{'MRR':>8}{'hit@1':>8}")
for r in rows:
    print(f"{r['config']:<15}{r['chunks']:>7}{r['recall@k']:>9.1%}{r['mrr']:>8.3f}{r['hit@1']:>8.1%}")

config          chunks  recall@5     MRR   hit@1
fixed-40/10       1174    71.7%   0.607   54.3%
fixed-60/15        787    71.7%   0.575   50.0%
fixed-100/20       448    71.7%   0.589   50.0%
sentence-40       1030    69.6%   0.524   43.5%
sentence-60        723    67.4%   0.568   50.0%
sentence-100       419    71.7%   0.513   39.1%


## 8. Retriever experiment

Same chunks across all three, BM25 as the baseline.

Worth noting what happened here. On an earlier six-document sample corpus, hybrid
RRF *lost* to plain BM25 by 6.7 points. On the full corpus it wins by 8.7. Rank
fusion needs enough documents to have something to fuse, and I'd have drawn the
opposite conclusion if I'd stopped at the small corpus.

In [19]:
chunks = chunk_sentences(docs, budget=60)
bm25, tfidf = BM25(chunks), TfidfRetriever(chunks)
hybrid = Hybrid([bm25, tfidf])

print(f"{'retriever':<12}{'recall@5':>10}{'MRR':>8}{'hit@1':>8}{'vs baseline':>14}")
base = None
for label, ret in [("BM25", bm25), ("TF-IDF", tfidf), ("hybrid RRF", hybrid)]:
    r = evaluate(ret, EVAL, k=5)
    if base is None:
        base = r["recall@k"]; delta = "baseline"
    else:
        delta = f"{r['recall@k'] - base:+.1%}"
    print(f"{label:<12}{r['recall@k']:>9.1%}{r['mrr']:>8.3f}{r['hit@1']:>8.1%}{delta:>14}")

retriever     recall@5     MRR   hit@1   vs baseline
BM25            67.4%   0.568   50.0%      baseline
TF-IDF          73.9%   0.627   56.5%         +6.5%
hybrid RRF      76.1%   0.632   54.3%         +8.7%


## 9. Failure analysis

The aggregate number hides which questions failed and why, so this prints the
individual misses and splits recall by question group.

This is the section the four-way grouping exists for.

In [20]:
r = evaluate(hybrid, EVAL, k=5)

print(f"{len(r['misses'])} misses of {r['n']}:\n")
for q, gold, got in r["misses"][:12]:
    print(f"  Q: {q}")
    print(f"     wanted {gold}, got {got}\n")

OVERLAP = {q for q, _ in EVAL[:16]}
PARA    = {q for q, _ in EVAL[16:35]}
INDIR   = {q for q, _ in EVAL[35:46]}

print("recall@5 by question type:")
for label, group in [("keyword overlap", OVERLAP), ("paraphrased", PARA), ("indirect", INDIR)]:
    sub = [(q, d) for q, d in EVAL if q in group and d]
    print(f"  {label:<18}{evaluate(hybrid, sub, k=5)['recall@k']:.0%}  (n={len(sub)})")

11 misses of 46:

  Q: Why does my browser block calls to my API from another site?
     wanted tutorial_cors, got ['async', 'tutorial_body', 'tutorial_testing']

  Q: How do I avoid repeating the same auth check in every endpoint?
     wanted tutorial_dependencies_index, got ['async', 'tutorial_bigger-applications', 'tutorial_response-model']

  Q: My project is one huge file, how do I break it up?
     wanted tutorial_bigger-applications, got ['async', 'deployment_docker', 'async']

  Q: How do I read a value the client stored in the browser and sends back?
     wanted tutorial_cookie-params, got ['tutorial_body', 'deployment_https', 'tutorial_security_first-steps']

  Q: How do I put a breakpoint in and step through my code?
     wanted tutorial_debugging, got ['deployment_docker', 'deployment_https', 'deployment_docker']

  Q: How do I check my code works without starting a server?
     wanted tutorial_testing, got ['tutorial_bigger-applications', 'deployment_https', 'tutorial_secu

## 10. Unanswerable questions

Six of the questions have no answer in the corpus, and the retriever still
returns its top 5 for every one of them. Retrieval always returns something —
deciding the results are too weak to use has to happen somewhere else.

The obvious candidate is a score threshold, so I checked whether the score ranges
for answerable and unanswerable questions actually separate.

In [21]:
def top_score(ret, q):
    hits = ret.search(q, k=1)
    return hits[0][1] if hits else 0.0

ans = [top_score(bm25, q) for q, d in EVAL if d]
una = [top_score(bm25, q) for q, d in EVAL if not d]

print(f"answerable   top BM25 score  mean {np.mean(ans):.2f}   min {min(ans):.2f}")
print(f"unanswerable top BM25 score  mean {np.mean(una):.2f}   max {max(una):.2f}")
print(f"\nseparable by a threshold: {min(ans) > max(una)}")
for q, d in EVAL:
    if not d:
        print(f"  unanswerable: {q!r} -> top score {top_score(bm25, q):.2f}")

answerable   top BM25 score  mean 14.67   min 8.55
unanswerable top BM25 score  mean 12.79   max 20.16

separable by a threshold: False
  unanswerable: 'How do I deploy to Kubernetes with a Helm chart?' -> top score 13.29
  unanswerable: 'What is the pricing for FastAPI enterprise support?' -> top score 8.05
  unanswerable: 'How do I train a machine learning model on this data?' -> top score 20.16
  unanswerable: 'How do I configure Redis caching layers?' -> top score 10.79
  unanswerable: 'What is the recommended way to write Django migrations?' -> top score 10.35
  unanswerable: 'How do I set up a GraphQL subscription?' -> top score 14.09


## 11. Summary

Every configuration in one table.

In [22]:
configs = [
    ("fixed-60/15  + BM25",   chunk_fixed,     BM25,            dict(size=60, overlap=15)),
    ("sentence-60  + BM25",   chunk_sentences, BM25,            dict(budget=60)),
    ("sentence-100 + BM25",   chunk_sentences, BM25,            dict(budget=100)),
    ("sentence-60  + TF-IDF", chunk_sentences, TfidfRetriever,  dict(budget=60)),
]

print(f"{'configuration':<24}{'chunks':>8}{'recall@5':>11}{'MRR':>8}{'hit@1':>8}")
print("-" * 59)
best = None
for label, chunker, cls, kw in configs:
    ch = chunker(docs, **kw)
    res = evaluate(cls(ch), EVAL, k=5)
    print(f"{label:<24}{len(ch):>8}{res['recall@k']:>10.1%}{res['mrr']:>8.3f}{res['hit@1']:>8.1%}")
    if best is None or res["mrr"] > best[1]:
        best = (label, res["mrr"])

# hybrid uses the best-performing chunking
ch = chunk_sentences(docs, budget=60)
res = evaluate(Hybrid([BM25(ch), TfidfRetriever(ch)]), EVAL, k=5)
print(f"{'sentence-60  + hybrid':<24}{len(ch):>8}{res['recall@k']:>10.1%}{res['mrr']:>8.3f}{res['hit@1']:>8.1%}")

print(f"\nbest MRR: {best[0]} ({best[1]:.3f})")
print(f"corpus: {len(docs)} documents, {sum(len(d.text.split()) for d in docs):,} words")
print(f"eval set: {len(EVAL)} questions, {sum(1 for _, d in EVAL if d)} answerable")

configuration             chunks   recall@5     MRR   hit@1
-----------------------------------------------------------
fixed-60/15  + BM25          787     71.7%   0.575   50.0%
sentence-60  + BM25          723     67.4%   0.568   50.0%
sentence-100 + BM25          419     71.7%   0.513   39.1%
sentence-60  + TF-IDF        723     73.9%   0.627   56.5%
sentence-60  + hybrid        723     76.1%   0.632   54.3%

best MRR: sentence-60  + TF-IDF (0.627)
corpus: 25 documents, 35,115 words
eval set: 52 questions, 46 answerable


## 12. Dense retrieval

Everything above is lexical. BM25 and TF-IDF both match on shared words, which is
why the paraphrased group fell to 53%. Dense retrieval embeds the question and each
chunk into one vector space and ranks by cosine similarity, so "why does my browser
block calls to my API from another site" can land next to a chunk about CORS without
sharing a single word with it.

Same chunks (sentence-60), same 52 questions, same metrics as every other retriever
here. Only the ranker changes. The model is `all-MiniLM-L6-v2` (22M parameters, a few
minutes on CPU for 723 chunks).

In [23]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

def encode(texts):
    """L2-normalised embeddings, so a dot product is cosine similarity."""
    return model.encode(texts, batch_size=64, normalize_embeddings=True,
                        show_progress_bar=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Mohit\anaconda4\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mohit\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
class DenseRetriever:
    """Embeds every chunk once, ranks by cosine similarity to the query."""
    def __init__(self, chunks, encode_fn):
        self.chunks = chunks
        self.encode = encode_fn
        self.M = np.asarray(encode_fn([c.text for c in chunks]))

    def search(self, query, k=5):
        q = np.asarray(self.encode([query]))[0]
        sims = self.M @ q
        order = np.argsort(sims)[::-1][:k]
        return [(self.chunks[i], float(sims[i])) for i in order]

chunks = chunk_sentences(docs, budget=60)
bm25, tfidf = BM25(chunks), TfidfRetriever(chunks)
dense = DenseRetriever(chunks, encode)
print(f"{len(chunks)} chunks embedded, dimension {dense.M.shape[1]}")

723 chunks embedded, dimension 384


## 13. Dense vs lexical, same chunks

BM25 stays the declared baseline. The hybrids use the same reciprocal rank fusion as
section 5, so fusing in the dense ranker is the only difference.

In [25]:
retrievers = [
    ("BM25 (baseline)",    bm25),
    ("TF-IDF",             tfidf),
    ("hybrid BM25+TF-IDF", Hybrid([bm25, tfidf])),
    ("dense",              dense),
    ("hybrid BM25+dense",  Hybrid([bm25, dense])),
    ("hybrid all three",   Hybrid([bm25, tfidf, dense])),
]

print(f"{'retriever':<22}{'recall@5':>10}{'MRR':>8}{'hit@1':>8}{'vs baseline':>14}")
base = None
for label, ret in retrievers:
    r = evaluate(ret, EVAL, k=5)
    if base is None:
        base = r["recall@k"]; delta = "baseline"
    else:
        delta = f"{r['recall@k'] - base:+.1%}"
    print(f"{label:<22}{r['recall@k']:>9.1%}{r['mrr']:>8.3f}{r['hit@1']:>8.1%}{delta:>14}")

retriever               recall@5     MRR   hit@1   vs baseline
BM25 (baseline)           67.4%   0.568   50.0%      baseline
TF-IDF                    73.9%   0.627   56.5%         +6.5%
hybrid BM25+TF-IDF        76.1%   0.632   54.3%         +8.7%
dense                     87.0%   0.766   69.6%        +19.6%
hybrid BM25+dense         84.8%   0.712   60.9%        +17.4%
hybrid all three          82.6%   0.710   63.0%        +15.2%


## 14. Does dense close the paraphrase gap?

This is the question section 9 left open. Recall by question group, for every
retriever, so the paraphrased column can be read directly against the 53% lexical
result.

In [26]:
print(f"{'retriever':<22}{'overlap':>10}{'paraphrased':>13}{'indirect':>10}")
for label, ret in retrievers:
    row = []
    for group in (OVERLAP, PARA, INDIR):
        sub = [(q, d) for q, d in EVAL if q in group and d]
        row.append(evaluate(ret, sub, k=5)["recall@k"])
    print(f"{label:<22}{row[0]:>10.0%}{row[1]:>13.0%}{row[2]:>10.0%}")
print("\nn: overlap=16  paraphrased=19  indirect=11")

retriever                overlap  paraphrased  indirect
BM25 (baseline)             100%          32%       82%
TF-IDF                      100%          58%       64%
hybrid BM25+TF-IDF          100%          53%       82%
dense                       100%          74%       91%
hybrid BM25+dense           100%          74%       82%
hybrid all three            100%          68%       82%

n: overlap=16  paraphrased=19  indirect=11


## 15. Can a cosine threshold abstain?

Section 10 showed BM25 scores can't separate answerable from unanswerable questions.
Cosine similarity is bounded and often better behaved, so it gets the same check
instead of assuming it fails the same way.

In [27]:
ans = [top_score(dense, q) for q, d in EVAL if d]
una = [top_score(dense, q) for q, d in EVAL if not d]

print(f"answerable   top cosine  mean {np.mean(ans):.3f}   min {min(ans):.3f}")
print(f"unanswerable top cosine  mean {np.mean(una):.3f}   max {max(una):.3f}")
print(f"\nseparable by a threshold: {min(ans) > max(una)}")
for q, d in EVAL:
    if not d:
        print(f"  unanswerable: {q!r} -> top cosine {top_score(dense, q):.3f}")

answerable   top cosine  mean 0.558   min 0.346
unanswerable top cosine  mean 0.414   max 0.488

separable by a threshold: False
  unanswerable: 'How do I deploy to Kubernetes with a Helm chart?' -> top cosine 0.437
  unanswerable: 'What is the pricing for FastAPI enterprise support?' -> top cosine 0.488
  unanswerable: 'How do I train a machine learning model on this data?' -> top cosine 0.387
  unanswerable: 'How do I configure Redis caching layers?' -> top cosine 0.424
  unanswerable: 'What is the recommended way to write Django migrations?' -> top cosine 0.434
  unanswerable: 'How do I set up a GraphQL subscription?' -> top cosine 0.315


## What I found

**Hybrid retrieval beat the BM25 baseline by 8.7 points**, 67.4% to 76.1%
recall@5. The same configuration lost by 6.7 points on a smaller sample corpus,
which isolates corpus scale rather than the method as the reason.

**Lexical retrieval collapses on paraphrase.** 100% recall when a question reuses
the document's vocabulary, 53% when it doesn't, 82% on indirect questions. The
53% group is the one that looks like real user queries.

**No score threshold separates answerable from unanswerable questions.** One
question the corpus can't answer scored 20.16, above the 14.67 mean for answerable
ones. Abstention needs a reranker or a sufficiency check, not a cutoff.

## Limitations

- Labels are document-level, so recall measures whether the right document came
  back, not the right passage. Chunk-level labels would be more precise and
  considerably more work to produce.
- I wrote the eval set, so it carries my bias toward questions I thought to ask.
  Real query logs would be messier and more representative.
- No reranking stage. A cross-encoder over the top 20 is likely the largest single
  improvement available here and is the next thing I'd add.
- Lexical retrieval only. Dense embeddings would probably close much of the
  paraphrase gap, which section 9 quantifies at 47 points.

In [ ]:
import os
os.environ.update(LANGFUSE_PUBLIC_KEY="pk-lf-local", LANGFUSE_SECRET_KEY="sk-lf-local", LANGFUSE_HOST="http://localhost:3000")
from langfuse import Langfuse
lf = Langfuse()
for q, gold in EVAL:
    with lf.start_as_current_observation(name="retrieve", input=q) as obs:
        ids = [c.doc_id for c, _ in hybrid.search(q, k=5)]
        obs.update(output=ids)
        if gold: obs.score_trace(name="hit@5", value=int(gold in ids))
lf.flush()